# CS_05.ipynb

**Aim:** Train an SVM classifier on grayscale image pixels to distinguish real and fake images.

Run cells from top to bottom. Dataset-dependent experiments include clear setup instructions.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Configure these folders for your machine. Expected structure:
# dataset/train/Real and dataset/train/Fake
IMG_SIZE = (64, 64)
REAL_DIR = os.path.join(r"C:\Users\SAISH MALVANKAR\05_real_and_fake_face-20261019T050939Z-1-001\05_real_and_fake_face", "training_real")
FAKE_DIR = os.path.join(r"C:\Users\SAISH MALVANKAR\05_real_and_fake_face-20261019T050939Z-1-001\05_real_and_fake_face", "training_fake")

In [3]:
def load_data(real_dir, fake_dir, img_size):
    images, labels = [], []
    for folder, label in [(real_dir, 1), (fake_dir, 0)]:
        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            continue
        for filename in sorted(os.listdir(folder)):
            path = os.path.join(folder, filename)
            image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            image = cv2.resize(image, img_size).astype(np.float32) / 255.0
            images.append(image.flatten())
            labels.append(label)
    if not images:
        return np.empty((0, img_size[0] * img_size[1]), dtype=np.float32), np.array([], dtype=int)
    return np.asarray(images), np.asarray(labels)
X, y = load_data(REAL_DIR, FAKE_DIR, IMG_SIZE)
print("X shape:", X.shape, "y shape:", y.shape)
if len(y):
    print("Fake:", int(np.sum(y == 0)), "Real:", int(np.sum(y == 1)))

X shape: (2041, 4096) y shape: (2041,)
Fake: 960 Real: 1081


In [4]:
if len(y) < 4 or len(np.unique(y)) < 2:
    print("Add at least two images in each class to the configured folders before training.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = make_pipeline(StandardScaler(), SVC(kernel="linear", class_weight="balanced"))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print(classification_report(y_test, pred, labels=[0,1], target_names=["Fake","Real"], zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred, labels=[0,1]))

Accuracy: 0.577
              precision    recall  f1-score   support

        Fake       0.55      0.57      0.56       192
        Real       0.61      0.58      0.59       217

    accuracy                           0.58       409
   macro avg       0.58      0.58      0.58       409
weighted avg       0.58      0.58      0.58       409

Confusion matrix:
 [[110  82]
 [ 91 126]]


In [6]:
# Predict one image after a model has been trained.
test_image_path = r"C:\Users\SAISH MALVANKAR\05_real_and_fake_face-20261019T050939Z-1-001\05_real_and_fake_face\training_real\real_00001.jpg"
if os.path.exists(test_image_path) and "model" in globals():
    image = cv2.imread(test_image_path, cv2.IMREAD_GRAYSCALE)
    if image is not None:
        image = cv2.resize(image, IMG_SIZE).astype(np.float32).flatten() / 255.0
        label = model.predict([image])[0]
        print("Prediction:", "Real" if label == 1 else "Fake")
else:
    print("Optional: place a test image at", test_image_path)

Prediction: Real
